# Download dataset to Google cloud bucket
>For the convenice run this on Google Colab 

## setup

0. If run locally download GCP cli tool first. The instruction can be accessed by [Install the Google Cloud CLI](https://docs.cloud.google.com/sdk/docs/install-sdk)

1. In terminal run


```
gcloud auth login
```
The instruction will be presented, follow it to complete the setup.

2. To setup your project space and setting billing account run

```
gcloud projects create --name="<>"

clound config set project <YOUR-PROJECT_ID>

gcloud billing project link <YOUR-PROJECT_ID> --billing-account=<YOUR_BILLING_ACCOUNT_ID>
```

3. Enable the Google cloud storage service by running
```
gcloud services enable dataproc.googleapis.com storage.googleapis.com
```
and create your bucket by
```
gcloud storage buckets create gs://<YOUR_BUCKET_NAME> \
  --location=asia-southeast1 \
  --uniform-bucket-level-access
```




In [ ]:
import requests
from google.colab import userdata
import os

In [ ]:
def get_download_url(session: requests.Session, dataset_id: str):
    init = session.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download",
    )

    if init.status_code != 201:
        print(f"Error initiating download for dataset {dataset_id}: HTTP {init.status_code}")
        return

    response = session.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download",
    )

    if response.status_code == 201:
        res_json = response.json()
        if res_json.get("data") is not None and "url" in res_json["data"]:
            return(res_json["data"]["url"])
        else:
            print(f"'data' or 'url' missing for dataset {dataset_id}. Response: {res_json}")
    else:
        print(f"Error fetching dataset {dataset_id}: HTTP {response.status_code}")



In [ ]:
COLLECTION_ID = 2279
BUCKET = f"{userdata.get('DSA5208_GS_BUCKET')}/raw"
API_KEY = userdata.get('DATAGOVSG')
S = requests.Session()
S.headers.update({
    "x-api-key": API_KEY,
    "Content-Type":"application/json"
})


In [ ]:
res = requests.get(
    f"https://api-production.data.gov.sg/v2/public/api/collections/{COLLECTION_ID}/metadata"
)
dataset_ids = res.json()["data"]["collectionMetadata"]["childDatasets"][1:]

In [ ]:
for i, dataset_id in enumerate(dataset_ids):
    print(f"Downloading {i+1}/{len(dataset_ids)}, ID: {dataset_id}")
    url = get_download_url(S, dataset_id)
    path = f"/tmp/rainfall_across_sg_{2017+i}.csv"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    os.system(f"gsutil cp {path} {BUCKET}/ && rm {path}")